# 03 — Split 128-dim Embeddings (All Groups)

Splits all **128-dim** embedding parquets (v1, v2, v3) into two halves **by rows**, preserving all 128 embedding columns in each half. Also merges and verifies reconstruction.

Original parquets are **never modified**.

| Original | Part A (first half of rows) | Part B (second half of rows) |
|---|---|---|
| `graphsage_v1_128_srisk_dataset.parquet` | `graphsage_v1_128a_srisk_dataset.parquet` | `graphsage_v1_128b_srisk_dataset.parquet` |
| `node2vec_v1_128_srisk_dataset.parquet` | `node2vec_v1_128a_srisk_dataset.parquet` | `node2vec_v1_128b_srisk_dataset.parquet` |
| `graphsage_v2_128_srisk_dataset.parquet` | `graphsage_v2_128a_srisk_dataset.parquet` | `graphsage_v2_128b_srisk_dataset.parquet` |
| `node2vec_v2_128_srisk_dataset.parquet` | `node2vec_v2_128a_srisk_dataset.parquet` | `node2vec_v2_128b_srisk_dataset.parquet` |
| `graphsage_v3_128_srisk_nolog_dataset.parquet` | `graphsage_v3_128a_srisk_nolog_dataset.parquet` | `graphsage_v3_128b_srisk_nolog_dataset.parquet` |
| `node2vec_v3_128_srisk_nolog_dataset.parquet` | `node2vec_v3_128a_srisk_nolog_dataset.parquet` | `node2vec_v3_128b_srisk_nolog_dataset.parquet` |

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path().resolve().parents[1]
EMB_DIR      = PROJECT_ROOT / 'src' / 'data' / 'embeddings'

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


## Configuration

In [2]:
TARGETS = [
    # Group 1 — v1 (link prediction, log target)
    (
        'graphsage_v1_128_srisk_dataset.parquet',
        'graphsage_v1_128a_srisk_dataset.parquet',
        'graphsage_v1_128b_srisk_dataset.parquet',
    ),
    (
        'node2vec_v1_128_srisk_dataset.parquet',
        'node2vec_v1_128a_srisk_dataset.parquet',
        'node2vec_v1_128b_srisk_dataset.parquet',
    ),
    # Group 2 — v2 (reconstruction loss, log target)
    (
        'graphsage_v2_128_srisk_dataset.parquet',
        'graphsage_v2_128a_srisk_dataset.parquet',
        'graphsage_v2_128b_srisk_dataset.parquet',
    ),
    (
        'node2vec_v2_128_srisk_dataset.parquet',
        'node2vec_v2_128a_srisk_dataset.parquet',
        'node2vec_v2_128b_srisk_dataset.parquet',
    ),
    # Group 3 — v3 (reconstruction loss, no-log target)
    (
        'graphsage_v3_128_srisk_nolog_dataset.parquet',
        'graphsage_v3_128a_srisk_nolog_dataset.parquet',
        'graphsage_v3_128b_srisk_nolog_dataset.parquet',
    ),
    (
        'node2vec_v3_128_srisk_nolog_dataset.parquet',
        'node2vec_v3_128a_srisk_nolog_dataset.parquet',
        'node2vec_v3_128b_srisk_nolog_dataset.parquet',
    ),
]

META_COLS = ['bank_id', 'year', 'quarter', 'period', 'systemic_risk_label']

## Split

In [3]:
for orig, out_a, out_b in TARGETS:
    src = EMB_DIR / orig
    df  = pd.read_parquet(src)

    emb_cols = [c for c in df.columns if c.startswith('emb_')]
    assert len(emb_cols) == 128, f'Expected 128 emb cols, got {len(emb_cols)}'

    half  = len(df) // 2
    df_a  = df.iloc[:half].reset_index(drop=True)
    df_b  = df.iloc[half:].reset_index(drop=True)

    df_a.to_parquet(EMB_DIR / out_a, index=False)
    df_b.to_parquet(EMB_DIR / out_b, index=False)

    print(f'{orig}  ({len(df)} rows, {len(emb_cols)} emb cols)')
    print(f'  A → {out_a}  ({len(df_a)} rows, {len(emb_cols)} emb cols)')
    print(f'  B → {out_b}  ({len(df_b)} rows, {len(emb_cols)} emb cols)')
    print()

graphsage_v1_128_srisk_dataset.parquet  (145536 rows, 128 emb cols)
  A → graphsage_v1_128a_srisk_dataset.parquet  (72768 rows, 128 emb cols)
  B → graphsage_v1_128b_srisk_dataset.parquet  (72768 rows, 128 emb cols)

node2vec_v1_128_srisk_dataset.parquet  (145536 rows, 128 emb cols)
  A → node2vec_v1_128a_srisk_dataset.parquet  (72768 rows, 128 emb cols)
  B → node2vec_v1_128b_srisk_dataset.parquet  (72768 rows, 128 emb cols)

graphsage_v2_128_srisk_dataset.parquet  (145536 rows, 128 emb cols)
  A → graphsage_v2_128a_srisk_dataset.parquet  (72768 rows, 128 emb cols)
  B → graphsage_v2_128b_srisk_dataset.parquet  (72768 rows, 128 emb cols)

node2vec_v2_128_srisk_dataset.parquet  (145536 rows, 128 emb cols)
  A → node2vec_v2_128a_srisk_dataset.parquet  (72768 rows, 128 emb cols)
  B → node2vec_v2_128b_srisk_dataset.parquet  (72768 rows, 128 emb cols)

graphsage_v3_128_srisk_nolog_dataset.parquet  (145536 rows, 128 emb cols)
  A → graphsage_v3_128a_srisk_nolog_dataset.parquet  (72768 rows

## Merge (reconstruct and verify)

In [4]:
for orig, out_a, out_b in TARGETS:
    df_orig = pd.read_parquet(EMB_DIR / orig)
    df_a    = pd.read_parquet(EMB_DIR / out_a)
    df_b    = pd.read_parquet(EMB_DIR / out_b)

    merged = pd.concat([df_a, df_b], ignore_index=True)

    emb_cols = sorted([c for c in df_orig.columns if c.startswith('emb_')],
                      key=lambda c: int(c.split('_')[1]))

    shape_match  = merged.shape == df_orig.shape
    values_match = np.allclose(df_orig[emb_cols].values, merged[emb_cols].values)

    print(f'{orig}')
    print(f'  merged shape : {merged.shape}  (original: {df_orig.shape})')
    print(f'  shape match  : {shape_match}')
    print(f'  values match : {values_match}')
    print()

graphsage_v1_128_srisk_dataset.parquet
  merged shape : (145536, 133)  (original: (145536, 133))
  shape match  : True
  values match : True

node2vec_v1_128_srisk_dataset.parquet
  merged shape : (145536, 133)  (original: (145536, 133))
  shape match  : True
  values match : True

graphsage_v2_128_srisk_dataset.parquet
  merged shape : (145536, 133)  (original: (145536, 133))
  shape match  : True
  values match : True

node2vec_v2_128_srisk_dataset.parquet
  merged shape : (145536, 133)  (original: (145536, 133))
  shape match  : True
  values match : True

graphsage_v3_128_srisk_nolog_dataset.parquet
  merged shape : (145536, 133)  (original: (145536, 133))
  shape match  : True
  values match : True

node2vec_v3_128_srisk_nolog_dataset.parquet
  merged shape : (145536, 133)  (original: (145536, 133))
  shape match  : True
  values match : True

